<a href="https://www.kaggle.com/code/mahsazamanifard/nlp-sarcastic-headline-detector?scriptVersionId=100089929" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# NLP sarcastic headline detector

**Here, we are going to classify headlines based on being sarcastic and use sentences of our own to evaluate the performance.**

**Note: The focus is on creating a model, so I'm not going to focus on clean the data.**

**For that, I have written a separate comprehensive notebook, Using [MachineLearningMastery](https://machinelearningmastery.com/) posts, you can check it out[ here!](https://www.kaggle.com/code/mahsazamanifard/cleaning-and-encoding-text-techniques-nlp)**

<img src="https://t4.ftcdn.net/jpg/03/80/35/39/360_F_380353964_T1BYrcnTa10esMxYRAlruj6OpyFkUufo.jpg" width="300">

# Load Data:
files are in json format, so we need to transform them into python format. the json object will become a list of dictionaries in python after transforming.

In [ ]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))
import numpy as np
import pandas as pd

In [ ]:
import json

f=open('../input/news-headlines-dataset-for-sarcasm-detection/Sarcasm_Headlines_Dataset_v2.json')
data = [json.loads(line) for line in f]


**NOTE:** we can't just use json.loads here, as there are multiple json objects in the file. [StackOverflow](https://stackoverflow.com/questions/21058935/python-json-loads-shows-valueerror-extra-data)

# Explore and Tokenize:

In [ ]:
data[:5]

In [ ]:
sentences,labels,urls=[],[],[]

In [ ]:
for item in data:
    sentences.append(item['headline'])
    labels.append(item['is_sarcastic'])
    urls.append(item['article_link'])

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
# Now let's separate our train and test set
training_size=20000
training_sentences=sentences[0:training_size]
testing_sentences=sentences[training_size:]

training_labels=labels[0:training_size]
testing_labels=labels[training_size:]

In [ ]:
#We are going to tokenize the words in our training set
# OOV means Out Of Vocabulary, which is for the words that are going to be out of our 10000 word vocabulary

tokenizer=Tokenizer(num_words=10000,oov_token="<OOV>")
tokenizer.fit_on_texts(training_sentences)

word_index=tokenizer.word_index

In [ ]:
# We can see the first 3 items of our word index
import itertools
dict(itertools.islice(word_index.items(), 0 ,3))

In [ ]:
training_sequences=tokenizer.texts_to_sequences(training_sentences)
training_padded=pad_sequences(training_sequences,maxlen=100,padding='post',truncating='post')

In [ ]:
training_padded[:2] 

In [ ]:
training_padded.shape

In [ ]:
testing_sequences=tokenizer.texts_to_sequences(testing_sentences)
testing_padded=pad_sequences(testing_sequences,maxlen=100,padding='post',truncating='post')

In [ ]:
testing_padded[:2]

In [ ]:
testing_padded.shape

**All of the  headlines are now encoded and have the same length, ready for being fed to the model! Let's now define the model**

# Define Model:

In [ ]:
import tensorflow as tf
vocab_size = 10000
embedding_dim = 16
max_length = 100
trunc_type='post'
padding_type='post'
oov_tok = "<OOV>"
training_size = 20000
model= tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size,embedding_dim,input_length=max_length),
    tf.keras.layers.GlobalAveragePooling1D(),
    tf.keras.layers.Dense(24,activation='relu'),
    tf.keras.layers.Dense(1,activation='sigmoid')
])

model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [ ]:
model.summary()

# Train:

In [ ]:
num_epochs = 30
history = model.fit(np.array(training_padded), np.array(training_labels), epochs=num_epochs, validation_data=(np.array(testing_padded), np.array(testing_labels)), verbose=2)

# Testing Performance:

In [ ]:
sentence=[
'Something terrible happened, just what I needed today!',
'the weather today is bright and sunny'
    
]

sequences=tokenizer.texts_to_sequences(sentence)
padded= pad_sequences(sequences, maxlen=max_length,padding=padding_type,truncating=trunc_type)
model.predict(padded)

**This means that the first sentence has a high probability of being sarcastic (.9) which is true, it is sacractic**
**And the second sentence has a really low prbability of being sarcastic, almost 0, which again is  true.** 

Reference: 
This [Playlist](https://youtube.com/playlist?list=PLQY2H8rRoyvzDbLUZkbudP-MFQZwNmU4S) On youtube, It's an absolutely great place for starting NLP!